# Concept-drift prediction with a Transformer

Inputs come from `data/annotation/{training,testing}/*.features.csv` — each row is a calendar window of one synthetic log, with **inter-case**, **resource-summary**, and **intra-case SOM state** features. The labels are five binary columns: one per drift perspective and an `drift_any` aggregate.

We slide a fixed-length window of consecutive rows through the model and ask it to predict, for the **last** position, whether a drift falls in that window.

## 1. Load and inspect the first dataset

In [2]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
TRAIN_DIR = ROOT / 'data' / 'annotation' / 'training'
TEST_DIR  = ROOT / 'data' / 'annotation' / 'testing'

train_files = sorted(TRAIN_DIR.glob('*.features.csv'))
test_files  = sorted(TEST_DIR.glob('*.features.csv'))
print(f'train logs: {len(train_files)}  test logs: {len(test_files)}')

first = pd.read_csv(train_files[0])
print(f"\nFirst log: {train_files[0].name}  shape: {first.shape}")
first.head()

train logs: 30  test logs: 8

First log: training_001.features.csv  shape: (120, 32)


,log_id,window_start,active_cases,new_arrivals,completions,total_events,mean_delta_t,std_delta_t,stalled_cases,res_events_total,...,intra_S4,intra_S5,intra_S6,intra_S7,intra_S8,drift_control_flow,drift_data,drift_inter_case,drift_resource,drift_any
0,training_001,2020-01-01 01:02:33.307892+00:00,17.0,17.0,15.0,98.0,2.440843,0.848652,0.0,98,...,0.173469,0.0,0.163265,0.000000,0.173469,0,0,0,0,0
1,training_001,2020-01-02 01:02:33.307892+00:00,17.0,15.0,17.0,94.0,2.330269,0.909248,0.0,94,...,0.159574,0.0,0.170213,0.000000,0.159574,0,0,0,0,0
2,training_001,2020-01-03 01:02:33.307892+00:00,12.0,12.0,11.0,69.0,2.402494,1.005327,0.0,69,...,0.173913,0.0,0.144928,0.014493,0.173913,0,0,0,0,0
3,training_001,2020-01-04 01:02:33.307892+00:00,15.0,14.0,14.0,83.0,2.395625,1.031490,0.0,83,...,0.156627,0.0,0.168675,0.000000,0.168675,0,0,0,0,0
4,training_001,2020-01-05 01:02:33.307892+00:00,15.0,14.0,14.0,84.0,2.223477,1.056774,0.0,84,...,0.166667,0.0,0.166667,0.000000,0.166667,0,0,0,0,0


Each row is one window. Columns split into three blocks of features (`active_cases…stalled_cases` — inter-case; `res_*` — resource summary; `intra_S*` — intra-case SOM state frequencies) and five label columns starting with `drift_`.

## 2. Concat all logs, split features/labels, standardise

In [3]:
import numpy as np

LABEL_COLS = [c for c in first.columns if c.startswith('drift_')]
FEATURE_COLS = [c for c in first.columns if c not in LABEL_COLS + ['log_id', 'window_start']]
print(f'{len(FEATURE_COLS)} feature columns, {len(LABEL_COLS)} label columns')

def stack(files):
    return [pd.read_csv(p) for p in files]

train_logs = stack(train_files)
test_logs  = stack(test_files)

# Fit a per-feature z-score on training rows; reuse on test.
all_train = pd.concat(train_logs, ignore_index=True)
mu  = all_train[FEATURE_COLS].mean().to_numpy()
sig = all_train[FEATURE_COLS].std().replace(0, 1).to_numpy()
print(f'training rows: {len(all_train)}')

25 feature columns, 5 label columns
training rows: 4481


## 3. Slice each log into fixed-length windowed sequences

The transformer takes `SEQ_LEN` consecutive windows and emits one label vector for the last position. We slide one step at a time.

In [4]:
SEQ_LEN = 16

def to_sequences(logs):
    Xs, Ys = [], []
    for df in logs:
        X = (df[FEATURE_COLS].to_numpy() - mu) / sig
        Y = df[LABEL_COLS].to_numpy().astype(np.float32)
        for i in range(SEQ_LEN, len(df) + 1):
            Xs.append(X[i - SEQ_LEN:i])
            Ys.append(Y[i - 1])
    return np.asarray(Xs, dtype=np.float32), np.asarray(Ys, dtype=np.float32)

X_train, Y_train = to_sequences(train_logs)
X_test,  Y_test  = to_sequences(test_logs)
print(f'X_train: {X_train.shape}  Y_train: {Y_train.shape}')
print(f'X_test : {X_test.shape}   Y_test : {Y_test.shape}')
print(f'positive fraction (drift_any): {Y_train[:, -1].mean():.4f}')

X_train: (4031, 16, 25)  Y_train: (4031, 5)
X_test : (1082, 16, 25)   Y_test : (1082, 5)
positive fraction (drift_any): 0.0131


Drift change-points are sparse, so positive labels are rare — we'll weight the loss to compensate.

## 4. Transformer with one prediction head per drift type

A tiny encoder-only transformer: input → linear projection → 2 self-attention blocks → take the last position → one linear head per label.

In [5]:
import torch
import torch.nn as nn

D_MODEL  = 64
N_HEADS  = 4
N_LAYERS = 2

class DriftTransformer(nn.Module):
    def __init__(self, n_features, n_labels, seq_len):
        super().__init__()
        self.input_proj = nn.Linear(n_features, D_MODEL)
        self.pos = nn.Parameter(torch.zeros(1, seq_len, D_MODEL))
        layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=N_HEADS, dim_feedforward=128,
            dropout=0.1, batch_first=True, activation='gelu',
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=N_LAYERS)
        # one independent head per drift perspective
        self.heads = nn.ModuleList([nn.Linear(D_MODEL, 1) for _ in range(n_labels)])

    def forward(self, x):
        h = self.input_proj(x) + self.pos
        h = self.encoder(h)[:, -1]  # last-position representation
        return torch.cat([head(h) for head in self.heads], dim=-1)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = DriftTransformer(len(FEATURE_COLS), len(LABEL_COLS), SEQ_LEN).to(device)
print(model)
print(f'\nparameters: {sum(p.numel() for p in model.parameters()):,}  device: {device}')

DriftTransformer(
  (input_proj): Linear(in_features=25, out_features=64, bias=True)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (heads): ModuleList(
    (0-4): 5 x Linear(in_features=64, out_features=1, bias=True)
  )
)

parameters: 69,957  device: cpu


## 5. Training loop

In [6]:
from torch.utils.data import DataLoader, TensorDataset

EPOCHS = 12
BATCH  = 64
LR     = 1e-3

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(Y_train))
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True)

# pos_weight = (#neg / #pos) per label, capped so we don't explode on empty columns
pos = Y_train.sum(axis=0)
neg = len(Y_train) - pos
pos_weight = torch.tensor(np.clip(neg / np.maximum(pos, 1), 1, 200), dtype=torch.float32, device=device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optim = torch.optim.AdamW(model.parameters(), lr=LR)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total, n_batches = 0.0, 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = loss_fn(logits, yb)
        optim.zero_grad()
        loss.backward()
        optim.step()
        total += loss.item()
        n_batches += 1
    print(f'epoch {epoch:02d}  loss {total / n_batches:.4f}')

epoch 01  loss 1.2770
epoch 02  loss 1.1193
epoch 03  loss 1.0782
epoch 04  loss 0.9724
epoch 05  loss 0.9487
epoch 06  loss 0.8010
epoch 07  loss 0.7935
epoch 08  loss 0.6530
epoch 09  loss 0.6285
epoch 10  loss 0.5900
epoch 11  loss 0.5943
epoch 12  loss 0.4802


## 6. Test evaluation

Report per-label accuracy, ROC-AUC, and a confusion summary on the held-out logs.

In [7]:
from sklearn.metrics import roc_auc_score

model.eval()
with torch.no_grad():
    logits = model(torch.from_numpy(X_test).to(device)).cpu().numpy()
probs = 1 / (1 + np.exp(-logits))
preds = (probs > 0.5).astype(int)

for i, name in enumerate(LABEL_COLS):
    y, p = Y_test[:, i], probs[:, i]
    acc = (preds[:, i] == y).mean()
    pos = int(y.sum())
    auc = roc_auc_score(y, p) if 0 < pos < len(y) else float('nan')
    print(f'{name:22s}  acc={acc:.3f}  auc={auc:.3f}  positives={pos}/{len(y)}')

drift_control_flow      acc=0.820  auc=0.375  positives=6/1082
drift_data              acc=0.872  auc=0.084  positives=1/1082
drift_inter_case        acc=0.860  auc=0.756  positives=4/1082
drift_resource          acc=0.807  auc=0.650  positives=5/1082
drift_any               acc=0.661  auc=0.480  positives=16/1082
